# Phase 1 Base vs SFT Evaluation (Colab)

This notebook calls the `python -m finpost.evals.eval_exact` CLI from issue 02 to run exact-answer evaluation on one or more checkpoints. It renders the accuracy and cost summaries inline.

Platform setup (Drive mount for Colab, Kaggle Secrets, etc.) is handled in cells 2 and 4. All eval logic lives in the CLI; the notebook just invokes it via shell escape and loads the resulting artifacts back.

**Links:**
- [Issue 03 PRD](./.scratch/phase1-base-vs-sft-eval/README.md)
- [Issue 02 CLI](./.scratch/phase1-base-vs-sft-eval/issues/02-eval-exact-cli-and-output.md)


In [ ]:
import torch

!nvidia-smi

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set up paths
REPO_ROOT = '/content/finpost'
RESULTS_ROOT = '/content/drive/MyDrive/finpost_runs/results/evals'
CHECKPOINT_PATH = '/content/drive/MyDrive/finpost_runs/checkpoints/combined_hf_step_3000'

# Create results directory if needed
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'RESULTS_ROOT: {RESULTS_ROOT}')
print(f'CHECKPOINT_PATH: {CHECKPOINT_PATH}')


In [ ]:
import subprocess
import sys

# Clone or update the repo
if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', 'https://github.com/shannan-liu1/finpost.git', REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=False)

# Install in editable mode
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_ROOT], check=True)
print('Repository installed')


In [ ]:
# For Colab: assume checkpoint is already in Drive
# This cell verifies it exists and optionally unzips if needed

import os
from pathlib import Path

checkpoint_dir = Path(CHECKPOINT_PATH).parent
checkpoint_name = Path(CHECKPOINT_PATH).name

# Check if checkpoint exists as a directory
if os.path.isdir(CHECKPOINT_PATH):
    print(f'Checkpoint directory found: {CHECKPOINT_PATH}')
    print('Contents:')
    for item in os.listdir(CHECKPOINT_PATH)[:10]:
        print(f'  {item}')
else:
    # Check for a zip file
    zip_path = f'{CHECKPOINT_PATH}.zip'
    if os.path.isfile(zip_path):
        print(f'Found zip: {zip_path}, extracting...')
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(checkpoint_dir)
        print('Extraction complete')
    else:
        print(f'ERROR: Checkpoint not found at {CHECKPOINT_PATH} or {zip_path}')
        print('Please upload the checkpoint to Drive at the path shown above')


In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from finpost.evals.sources import REGISTRY

print('Available eval sources:')
for source_name in REGISTRY.keys():
    print(f'  {source_name}')


In [ ]:
# Call the eval_exact CLI
# This invokes the exact-answer evaluation and writes accuracy and cost summaries

cmd = [
    sys.executable, '-m', 'finpost.evals.eval_exact',
    '--checkpoints', f'base=Qwen/Qwen2.5-0.5B', f'combined={CHECKPOINT_PATH}',
    '--sources', 'gsm8k', 'math',
    '--n', '500',
    '--seed', '42',
    '--out-dir', RESULTS_ROOT,
    '--batch-size-gsm8k', '8',
    '--batch-size-math', '4',
]

# Optional: add GPU cost if you know your hourly rate
# cmd.extend(['--gpu-cost-per-hour', '1.39'])

import subprocess
result = subprocess.run(cmd, cwd=REPO_ROOT)
if result.returncode != 0:
    print(f'ERROR: eval_exact CLI exited with code {result.returncode}')
else:
    print('Evaluation complete')


In [ ]:
import json
from pathlib import Path

# Load accuracy_summary.json
accuracy_path = Path(RESULTS_ROOT) / 'accuracy_summary.json'
if accuracy_path.exists():
    with open(accuracy_path) as f:
        accuracy_data = json.load(f)

    # Try to render as a DataFrame if pandas is available
    try:
        import pandas as pd
        df = pd.DataFrame(accuracy_data)
        print('Accuracy Summary:')
        print(df.to_string())
    except ImportError:
        # Fallback: stdlib table
        print('Accuracy Summary (pandas not available):')
        if accuracy_data:
            # Print header
            headers = list(accuracy_data[0].keys())
            print(' | '.join(headers))
            print('-' * 80)
            for row in accuracy_data:
                print(' | '.join(str(row.get(h, '')) for h in headers))
else:
    print(f'accuracy_summary.json not found at {accuracy_path}')


In [ ]:
# Load cost_summary.json and print key metrics
cost_path = Path(RESULTS_ROOT) / 'cost_summary.json'
if cost_path.exists():
    with open(cost_path) as f:
        cost_data = json.load(f)

    print('Cost Summary:')
    print(f'  Start: {cost_data.get("start_time", "N/A")}')
    print(f'  End: {cost_data.get("end_time", "N/A")}')
    print(f'  Elapsed seconds: {cost_data.get("elapsed_sec", 0):.1f}')
    print(f'  GPU type: {cost_data.get("gpu_type", "N/A")}')
    print(f'  Generated tokens: {cost_data.get("total_generated_tokens", 0)}')
    print(f'  Tokens/sec: {cost_data.get("tokens_per_second", 0):.2f}')
    if cost_data.get("estimated_cost_usd") is not None:
        print(f'  Estimated cost: ${cost_data.get("estimated_cost_usd", 0):.4f}')
else:
    print(f'cost_summary.json not found at {cost_path}')
